# Task 1 — Article Type Classification

This notebook asks a simple question: which safe image-only method works best for the 124 article types? It starts with the data problem, then tests clear hypotheses. The default run is a small smoke check, not final evidence.


## 1. Problem and output

Input: one fashion image for one product `id`. Output: one label from the fixed 124-class `articleType` vocabulary. The final submission needs `id,gender,articleType,season,usage`, so this task fills only `articleType`.

We compare labelled development images only. The learned CNN starts from scratch. Pretrained weights are not part of the submitted model.


## 2. EDA evidence

The prepared evidence below describes image shape, grayscale files, class imbalance, and fold warnings. These are problems to solve. The choices in the table are hypotheses to test, not declared winners.


In [ ]:
from dataclasses import asdict

import pandas as pd

from fashion.config import DEVELOPMENT_CLASS_SUMMARY_CSV
from fashion.data.dataset import get_samples, load_label_maps, load_splits
from fashion.task1 import (
    build_task1_decision_evidence,
    build_task1_problem_profile,
)

TARGET = "articleType"
splits = load_splits()
task1_development = get_samples(splits, partition="development", target=TARGET)
article_type_map = load_label_maps()[TARGET]
article_type_classes = tuple(article_type_map["classes"])
class_summary = pd.read_csv(DEVELOPMENT_CLASS_SUMMARY_CSV, keep_default_na=False)
problem_profile = build_task1_problem_profile(splits, class_summary)

display(pd.DataFrame([asdict(problem_profile)]))
display(build_task1_decision_evidence(problem_profile))


## 3. Safety contract

`data/processed/splits.csv` is the only split. We do not make a new split. Development rows use the saved five folds. Holdout and quarantine labels stay sealed until Notebook 06.

Only image pixels are model input. Names, years, file size, and the other targets are not used because they can be shortcuts. Every physical training run is registered in `results/runs.csv`.


## 4. Evaluation

Each candidate uses all five saved folds. The main score is fixed-label macro-F1 across all 124 classes. Macro-F1 gives a rare class the same weight as a common class, so accuracy cannot hide rare-class failure.

We compare five-fold mean macro-F1 and its sample standard deviation. Per-class F1, out-of-fold predictions, confusion pairs, weighted F1, and accuracy explain the score; they do not replace the fixed decision rule.


## 5. Candidate hypotheses

A scratch CNN is the image-learning baseline. HOG shape features with two classic classifiers are useful comparison baselines. All candidates use the same sealed folds and fixed metrics.

Hypothesis: mild image augmentation improves five-fold mean macro-F1 without making folds much less stable. A classic candidate earns consideration only after its selected five-fold evidence is complete, not from a quick tuning result.


## 6. Controlled preprocessing

The EDA suggests a shared image contract: EXIF orientation, RGB conversion, shape-preserving 60-by-80 white padding, then fold-fitted normalization. This stops stretching and handles grayscale images the same way for every family.

The control has no random change. The second condition adds small seeded flips, rotations, movement, scale, brightness, and contrast changes. The model, folds, seed, budget, and metric stay fixed so the test isolates augmentation.


In [ ]:
from fashion.task1 import (
    DEFAULT_TASK1_PREPROCESSING,
    TASK1_CONTROL_PREPROCESSING,
    run_task1_classical_experiment,
    run_task1_experiment,
)

preprocessing_candidates = {
    "control_no_augmentation": TASK1_CONTROL_PREPROCESSING.to_dict(),
    "hypothesis_mild_augmentation": DEFAULT_TASK1_PREPROCESSING.to_dict(),
}
display(pd.DataFrame(preprocessing_candidates).T)


## 7. Run plan

| Stage | What it checks | Evidence it can support |
|---|---|---|
| CNN smoke | One registered fold-0 path | Setup only; not a winner |
| CNN full | Control and augmentation over five folds each | Augmentation hypothesis |
| Classic smoke | One safe path for each HOG candidate | Setup only; not a winner |
| Classic tune | Fold-0 HOG and settings selection | Settings selection only |
| Classic final | Selected candidates over five folds | Final classic comparison evidence |

The smoke defaults make Run All safe for a quick path check. Full and final runs are intentional separate work. Scores and run IDs come from registered evidence, never hand-written markdown.


## 8. Run controllers

The reusable Task 1 package owns fitting, metrics, and registration. This notebook only chooses a stage and displays the evidence returned by the controllers.


In [ ]:
RUN_MODE = "smoke"  # Change to "full" only for the ten registered CNN runs.
task1_experiment = run_task1_experiment(
    splits,
    article_type_map,
    mode=RUN_MODE,
)
display(task1_experiment.fold_metrics)
if not task1_experiment.comparison.empty:
    display(task1_experiment.comparison)
if not task1_experiment.oof_metrics.empty:
    display(task1_experiment.oof_metrics)


In [ ]:
CLASSICAL_STAGE = "smoke"  # Then use "tune" and finally "final" in separate runs.
classic_experiment = run_task1_classical_experiment(
    splits,
    article_type_map,
    stage=CLASSICAL_STAGE,
)
display(classic_experiment.fold_metrics)
if not classic_experiment.tuning.empty:
    display(classic_experiment.tuning)
if not classic_experiment.comparison.empty:
    display(classic_experiment.comparison)
if not classic_experiment.oof_metrics.empty:
    display(classic_experiment.oof_metrics)


In [ ]:
from fashion.config import TASK1_FIGURE_DIR
from fashion.task1 import write_task1_comparison_figure, write_task1_confusion_figure

if RUN_MODE == "full" and not task1_experiment.fold_metrics.empty:
    write_task1_comparison_figure(task1_experiment.fold_metrics)
    for candidate_id, predictions in task1_experiment.oof_predictions.items():
        write_task1_confusion_figure(
            predictions,
            article_type_classes,
            output=TASK1_FIGURE_DIR / f"cnn_oof_confusion_{candidate_id}.png",
        )

if CLASSICAL_STAGE == "final":
    for candidate_id, predictions in classic_experiment.oof_predictions.items():
        write_task1_confusion_figure(
            predictions,
            article_type_classes,
            output=TASK1_FIGURE_DIR / f"classical_oof_confusion_{candidate_id}.png",
        )


## 9. Results

Read result tables from the completed controllers. Smoke evidence proves only that the path works. Full CNN and final classic evidence are needed before comparing candidates. Do not copy old scores here: fresh registered tables are the source of any conclusion.


## 10. Failure analysis

The tables below use in-memory evidence returned by completed controllers. `example_ids` point to prepared development rows for representative-image inspection. They help find failures, but never choose or rank a model alone.


In [ ]:
from fashion.task1 import (
    build_task1_confusion_pairs,
    build_task1_weak_class_table,
)

per_class_evidence = {
    **{f"cnn:{key}": value for key, value in task1_experiment.per_class.items()},
    **{f"classic:{key}": value for key, value in classic_experiment.per_class.items()},
}
oof_prediction_evidence = {
    **{f"cnn:{key}": value for key, value in task1_experiment.oof_predictions.items()},
    **{f"classic:{key}": value for key, value in classic_experiment.oof_predictions.items()},
}

if per_class_evidence:
    display(build_task1_weak_class_table(per_class_evidence, limit=10))
else:
    print("Weak-class evidence is not ready; complete full/final runs first.")

if oof_prediction_evidence:
    display(build_task1_confusion_pairs(oof_prediction_evidence, limit=10))
else:
    print("Confusion-pair evidence is not ready; complete full/final runs first.")


## 11. Decision

The augmentation hypothesis is passed only if five-fold mean macro-F1 improves without an unacceptable increase in fold standard deviation. It has failed if that evidence does not appear.

A classic model earns consideration only from complete selected five-fold evidence, not its fold-0 tuning score. The final model is not chosen from accuracy alone; use macro-F1, fold stability, per-class failures, confusion pairs, and practical limitations.

If evidence files are absent, the handoff says NOT READY and names the missing run stage. Until then, the decision is not ready.


## 12. Final handoff

**NOT READY.** Before Notebook 06, complete the CNN `full` stage and the classic `tune` then `final` stages. Then record the selected run ID, preprocessing configuration, fixed metric, five-fold evidence, refit steps, output path, and honest limits.

After the evidence exists, update the Results, Failure analysis, and Decision sections from those fresh tables. Do not use holdout labels to make this development decision.
